### MTL 3 model results + few shot

In [1]:
import os 
import json
import pandas as pd

def get_result_table(result_path, caption):
    f1_list = []
    auc_list = []

    with open(result_path, 'r') as f:
        data = json.load(f)

        # Loop through each annotator in the file
        for i, annotator in enumerate(data['task_name']):
            # Append f1 and auc to respective lists
            f1_list.append({'Annotator': annotator, 'F1': data['f1'][i]})
            auc_list.append({'Annotator': annotator, 'AUC': data['auc'][i]})

    # Create dataframes for f1 and auc
    f1_df = pd.DataFrame(f1_list)
    auc_df = pd.DataFrame(auc_list)

    result_df = pd.merge(f1_df, auc_df, on='Annotator')
    overall_avg = {'Annotator': 'Overall', 'F1': result_df['F1'].mean(), 'AUC': result_df['AUC'].mean()}
    result_df = pd.concat([result_df, pd.DataFrame([overall_avg])], ignore_index=True)
    result_table = result_df.to_latex(caption=caption, index=False)
    result_table = result_table.replace('lrr', 'l|cc')
    return result_table

        

In [2]:

files_path = '../results/roberta-base/brexit/Hate/seed_0/budget_2352/mtl_3'
f1_list = []
auc_list = []

for root, subdirs, files in os.walk(files_path):
    # print(root)
    for subdir in subdirs:
        for file in files:
            if file == 'test_result.json':
                with open(os.path.join(root,file), 'r') as f:
                    data = json.load(f)

                    # Loop through each annotator in the file
                    for i, annotator in enumerate(data['task_name']):
                        # Append f1 and auc to respective lists
                        f1_list.append({'Annotator': annotator, 'F1': data['f1'][i]})
                        auc_list.append({'Annotator': annotator, 'AUC': data['auc'][i]})

# Create dataframes for f1 and auc
f1_df = pd.DataFrame(f1_list)
auc_df = pd.DataFrame(auc_list)

# Group by annotator and calculate the average
avg_f1_df = f1_df.groupby('Annotator')['F1'].mean().reset_index()
avg_auc_df = auc_df.groupby('Annotator')['AUC'].mean().reset_index()
# Concatenate t
std_f1_df = f1_df.groupby('Annotator')['F1'].std().reset_index()
std_auc_df = auc_df.groupby('Annotator')['AUC'].std().reset_index()
# he overall average row to the end

# Merge dataframes on annotator
result_df = pd.merge(avg_f1_df, avg_auc_df, on='Annotator')
std_df = pd.merge(std_f1_df, std_auc_df, on='Annotator')
# print(result_df)
# print(std_df)
result_df = pd.merge(result_df, std_df, on='Annotator', suffixes=('_avg', '_std'))
overall_avg = {'Annotator': 'Overall', 'F1_avg': result_df['F1_avg'].mean(), 'AUC_avg': result_df['AUC_avg'].mean(), 
               'F1_std': result_df['F1_avg'].std(), 'AUC_std': result_df['AUC_avg'].std()}


result_df = pd.concat([result_df, pd.DataFrame([overall_avg])], ignore_index=True)

# # Print the resulting dataframe
print(result_df.to_latex(caption='Results for MTl 3 Brexit Hate',
                         label='tab:brexit_hate'))

result_df

\begin{table}
\caption{Results for MTl 3 Brexit Hate}
\label{tab:brexit_hate}
\begin{tabular}{llrrrr}
\toprule
 & Annotator & F1_avg & AUC_avg & F1_std & AUC_std \\
\midrule
0 & Ann1 & 0.317907 & 0.757862 & 0.083725 & 0.078304 \\
1 & Ann2 & 0.366007 & 0.816247 & 0.068575 & 0.090094 \\
2 & Ann3 & 0.379762 & 0.791430 & 0.051072 & 0.058838 \\
3 & Ann4 & 0.587256 & 0.745960 & 0.052673 & 0.040333 \\
4 & Ann5 & 0.594463 & 0.737757 & 0.080803 & 0.053991 \\
5 & Ann6 & 0.559682 & 0.776733 & 0.055086 & 0.035024 \\
6 & Overall & 0.467513 & 0.770998 & 0.125965 & 0.029673 \\
\bottomrule
\end{tabular}
\end{table}



,Annotator,F1_avg,AUC_avg,F1_std,AUC_std
0,Ann1,0.317907,0.757862,0.083725,0.078304
1,Ann2,0.366007,0.816247,0.068575,0.090094
2,Ann3,0.379762,0.791430,0.051072,0.058838
3,Ann4,0.587256,0.745960,0.052673,0.040333
4,Ann5,0.594463,0.737757,0.080803,0.053991
5,Ann6,0.559682,0.776733,0.055086,0.035024
6,Overall,0.467513,0.770998,0.125965,0.029673


### all few shot results

In [3]:
few_shot_path = "../results/roberta-base/brexit/Hate/seed_0/budget_2352/few_shot/strategy_mv/sampler"

f1_list = []
auc_list = []

for root, subdirs, files in os.walk(few_shot_path):
    # print(root)

    for file in files:
        if file == 'test_result.json':
            with open(os.path.join(root,file), 'r') as f:
                data = json.load(f)
                # print(root)
                annotator = root.split('/')[-2]
                k_shot = root.split('/')[-3]    
                mtl = root.split('/')[-1]
                n_mtl = len(mtl.split('_')[1].split(","))
                # print(annotator)
                # Loop through each annotator in the file
                # for i, annotator in enumerate(data['task_name']):
                #     # Append f1 and auc to respective lists
                f1_list.append({'Annotator': annotator, 'F1': data['f1'], 'AUC': data['auc'] , 'k-shot' : k_shot, 'n_mtl': n_mtl, 'mtl': mtl})
             

# Create dataframes for f1 and auc
f1_df = pd.DataFrame(f1_list)


## Group by annotator and calculate the average
result_df = f1_df.groupby(['n_mtl', 'k-shot','Annotator']).agg({'F1': ['mean', 'std'], 'AUC' : ['mean','std']}).reset_index()

# result_df_std = f1_df.groupby(['n_mtl', 'k-shot','Annotator']).agg({'F1': 'std', 'AUC' : 'std'}).reset_index()
result_df.columns = [''.join(col).strip() for col in result_df.columns.values]



overall_avg = result_df.groupby(['n_mtl', 'k-shot']).agg({'F1mean': ['mean' , 'std'], 'AUCmean' : ['mean','std']}).reset_index()
overall_avg.columns = [''.join(col).strip() for col in overall_avg.columns.values]
overall_avg = overall_avg.rename(columns={'F1meanmean': 'F1mean', 'F1meanstd': 'F1std', 'AUCmeanmean': 'AUCmean', 'AUCmeanstd': 'AUCstd'})


result_df = result_df.round(3).reindex()
result_df['result'] = result_df['F1mean'].astype(str) + ' (' + result_df['F1std'].astype(str) + ')' + ' / ' + result_df['AUCmean'].astype(str) + ' (' + result_df['AUCstd'].astype(str) + ')'

overall_avg = overall_avg.round(3).reindex()
overall_avg['result'] = overall_avg['F1mean'].astype(str) + ' (' + overall_avg['F1std'].astype(str) + ')' + ' / ' + overall_avg['AUCmean'].astype(str) + ' (' + overall_avg['AUCstd'].astype(str) + ')'
overall_avg['Annotator'] = 'Total'

result_df = pd.concat([result_df, overall_avg], ignore_index=True)



# # print(result_df.to_latex(caption='Results Few shot on mtl Brexit Hate',
# #                          label='tab:brexit_hate'))

for i in [3,4,5]:

    table_df  = result_df[result_df['n_mtl'] == i]
    table_df = table_df[['k-shot', 'Annotator', 'result']]



    table_df = table_df.pivot(index= 'Annotator', columns='k-shot', values='result').reset_index()

    table_df = table_df[['Annotator', '16', '32', '64', '128']]

    latex_table = table_df.to_latex(caption=f'Results Few shot on {i} mtl Brexit Hate, selection method : balanced mv + weighted sampler',
                            label='tab:brexit_hate', index=False)
    latex_table = latex_table.replace('lllll', 'l|cccc')
    latex_table = latex_table.replace('Total', '\hline Total')
    latex_table = latex_table.replace('\midrule', '\hline \midrule')
    latex_table = latex_table.replace('\\begin{table}', '\\begin{table} \small')
    latex_table = latex_table.replace("\\begin{table}", '\\begin{table*}')
    latex_table = latex_table.replace('\end{table}', '\end{table*}')
    
    print(latex_table)


\begin{table*} \small
\caption{Results Few shot on 3 mtl Brexit Hate, selection method : balanced mv + weighted sampler}
\label{tab:brexit_hate}
\begin{tabular}{l|cccc}
\toprule
Annotator & 16 & 32 & 64 & 128 \\
\hline \midrule
Ann1 & 0.351 (0.1) / 0.661 (0.074) & 0.298 (0.126) / 0.624 (0.072) & 0.296 (0.062) / 0.625 (0.048) & 0.272 (0.123) / 0.614 (0.071) \\
Ann2 & 0.105 (0.083) / 0.536 (0.045) & 0.161 (0.155) / 0.559 (0.076) & 0.122 (0.138) / 0.54 (0.061) & 0.161 (0.198) / 0.563 (0.105) \\
Ann3 & 0.378 (0.119) / 0.67 (0.074) & 0.35 (0.158) / 0.639 (0.079) & 0.288 (0.106) / 0.618 (0.069) & 0.343 (0.111) / 0.624 (0.049) \\
Ann4 & 0.501 (0.114) / 0.693 (0.071) & 0.535 (0.059) / 0.707 (0.043) & 0.507 (0.078) / 0.694 (0.054) & 0.575 (0.036) / 0.742 (0.032) \\
Ann5 & 0.569 (0.103) / 0.72 (0.063) & 0.604 (0.069) / 0.741 (0.05) & 0.583 (0.08) / 0.731 (0.058) & 0.61 (0.067) / 0.752 (0.051) \\
Ann6 & 0.521 (0.076) / 0.737 (0.065) & 0.495 (0.101) / 0.714 (0.07) & 0.526 (0.063) / 0.73 (0.046) & 

### Full data  budget result

In [27]:
import json
result_path = '../results/roberta-base/brexit/Hate/seed_0/budget_4704/mtl_6/mtl_Ann1,Ann2,Ann3,Ann4,Ann5,Ann6_64/test_result.json'
caption = 'Results for full data Brexit Hate'

print(get_result_table(result_path, caption))

\begin{table}
\caption{Results for full data Brexit Hate}
\begin{tabular}{l|cc}
\toprule
Annotator & F1 & AUC \\
\midrule
Ann1 & 0.285714 & 0.785115 \\
Ann2 & 0.326531 & 0.843816 \\
Ann3 & 0.392157 & 0.859004 \\
Ann4 & 0.631579 & 0.772727 \\
Ann5 & 0.666667 & 0.778087 \\
Ann6 & 0.545455 & 0.768689 \\
Overall & 0.474684 & 0.801240 \\
\bottomrule
\end{tabular}
\end{table}



### Baseline mtl results

In [26]:
import json
result_path = '../results/roberta-base/brexit/Hate/seed_0/budget_2352/mtl_6/mtl_Ann1,Ann2,Ann3,Ann4,Ann5,Ann6_64/test_result.json'
caption = 'Results for baseline with 2353 budget (mtl on 6 annotators) Brexit Hate'

print(get_result_table(result_path, caption))

\begin{table}
\caption{Results for baseline with 2353 budget (mtl on 6 annotators) Brexit Hate}
\begin{tabular}{l|cc}
\toprule
Annotator & F1 & AUC \\
\midrule
Ann1 & 0.258065 & 0.665618 \\
Ann2 & 0.193548 & 0.606918 \\
Ann3 & 0.242424 & 0.624493 \\
Ann4 & 0.517241 & 0.681818 \\
Ann5 & 0.507937 & 0.671500 \\
Ann6 & 0.416667 & 0.650054 \\
Overall & 0.355980 & 0.650067 \\
\bottomrule
\end{tabular}
\end{table}

